In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [44]:

drive_path = '/content/drive/MyDrive/WIDER_FACE_DATA/'

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!cp "{drive_path}/WIDER_train.zip" .
!cp "{drive_path}/WIDER_val.zip" .
!cp "{drive_path}/wider_face_split.zip" .

!unzip -q WIDER_train.zip
!unzip -q WIDER_val.zip
!unzip -q wider_face_split.zip

import os
print("WIDER_train/images count:", len(os.listdir("WIDER_train/images")))
print("wider_face_split directory contents:", os.listdir("wider_face_split"))

KeyboardInterrupt: 

In [45]:
import os

def parse_wider(txt_path):
    data = []
    with open(txt_path) as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        img_name_candidate = lines[i].strip()

        if not img_name_candidate or not (img_name_candidate.lower().endswith('.jpg') or img_name_candidate.lower().endswith('.png')):
            i += 1
            continue

        img = img_name_candidate
        i += 1

        num_faces_str = ""
        if i < len(lines):
            num_faces_str = lines[i].strip()

        try:
            n = int(num_faces_str)
        except ValueError:

            data.append((img, []))
            continue

        i += 1

        boxes = []
        for _ in range(n):
            if i >= len(lines):
                print(f"Warning: Hết dòng '{img}' {n}")
                break

            box_line = lines[i].strip()
            if not box_line:
                i += 1
                continue

            line_parts = list(map(int, box_line.split()))
            if len(line_parts) >= 4:
                x, y, w, h = line_parts[:4]
                if w > 0 and h > 0:
                    boxes.append([x, y, w, h])
            i += 1

        if boxes:
            data.append((img, boxes))
    return data


In [46]:
dataset = parse_wider("wider_face_split/wider_face_train_bbx_gt.txt")
print(f"Parsed {len(dataset)} images")

Parsed 12876 images kèm annotations


In [47]:
from torch.utils.data import Dataset, DataLoader
import cv2
import torch
import numpy as np
import random

class WiderDataset(Dataset):
    def __init__(self, data, img_dir, img_size=320, S=20):
        self.data = data
        self.img_dir = img_dir
        self.img_size = img_size
        self.S = S # 320 / 16 = 20

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name, boxes = self.data[idx]
        path = os.path.join(self.img_dir, img_name)

        img = cv2.imread(path)
        if img is None:
            print(f"Warning: Could not load image {path}")
            return self.__getitem__((idx + 1) % len(self))

        original_h, original_w, _ = img.shape

        do_flip = False
        if random.random() < 0.5:
            do_flip = True
            img = cv2.flip(img, 1) # flip

        img = cv2.resize(img, (self.img_size, self.img_size))

        img = torch.from_numpy(img).permute(2,0,1).float() / 255.0


        target = torch.zeros((self.S, self.S, 5))

        for box in boxes:
            x, y, bw, bh = box

            if do_flip:
                x = original_w - (x + bw)

            cx_norm = (x + bw / 2) / original_w
            cy_norm = (y + bh / 2) / original_h
            w_norm = bw / original_w
            h_norm = bh / original_h

            grid_x = int(cx_norm * self.S)
            grid_y = int(cy_norm * self.S)

            if 0 <= grid_x < self.S and 0 <= grid_y < self.S:

                if target[grid_y, grid_x, 4] == 0:

                    box_x_in_cell = cx_norm * self.S - grid_x
                    box_y_in_cell = cy_norm * self.S - grid_y

                    target[grid_y, grid_x] = torch.tensor(
                        [box_x_in_cell, box_y_in_cell, w_norm, h_norm, 1.0]
                    , dtype=torch.float32)

        return img, target


In [48]:
import os

train_ds = WiderDataset(dataset, "WIDER_train/images", img_size=416, S=26)
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=os.cpu_count(),
    pin_memory=True
)
print(f"Number of batches: {len(train_loader)}")
print(f"Using {os.cpu_count()} workers for DataLoader.")

Number of batches: 403
Using 2 workers for DataLoader.


In [49]:
import torch.nn as nn
import torch

class DetectingModel(nn.Module):
    def __init__(self, S=20):
        super().__init__()
        self.S = S

        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,1,1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,1,1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,1,1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128,256,3,1,1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2) # 20x20
        )

        self.pred = nn.Conv2d(256, 5, 1)

    def forward(self, x):
        x = self.features(x)
        x = self.pred(x)
        x = torch.sigmoid(x)
        x = x.permute(0,2,3,1)
        return x


In [50]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = DetectingModel(S=26).to(device)
print(model)

DetectingModel(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=

In [51]:
import torch.nn.functional as F

def loss_fn(pred, target):

    objectness_loss = F.mse_loss(pred[..., 4], target[..., 4], reduction='sum')

    obj_mask = target[..., 4] == 1

    pred_boxes = pred[obj_mask][:, :4] # cx, cy, w, h
    target_boxes = target[obj_mask][:, :4]

    bbox_loss = F.mse_loss(pred_boxes, target_boxes, reduction='sum') if pred_boxes.numel() > 0 else 0.0
    total_loss = bbox_loss * 5 + objectness_loss

    return total_loss

In [52]:
import torch.optim as optim
from tqdm.notebook import tqdm
import os

def train_model(model, train_loader, epochs=20, lr=1e-3):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    print(f"Training on: {device}")

    global drive_path
    checkpoint_dir = os.path.join(drive_path, "checkpoints")
    os.makedirs(checkpoint_dir, exist_ok=True)

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        loop = tqdm(train_loader, leave=True, desc=f"Epoch {epoch+1}/{epochs}")

        for imgs, targets in loop:
            imgs, targets = imgs.to(device), targets.to(device)

            preds = model(imgs)
            loss = loss_fn(preds, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            loop.set_postfix(loss=total_loss / (loop.n + 1))

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.4f}")

        if (epoch + 1) % 5 == 0:
            checkpoint_path = os.path.join(checkpoint_dir, f"model_epoch_{epoch+1}.pth")
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Saved model checkpoint to {checkpoint_path}")




In [53]:
train_model(model, train_loader, epochs=80, lr=1e-3)

Training on: cuda


Epoch 1/80:   0%|          | 0/403 [00:00<?, ?it/s]

KeyboardInterrupt: 